# Testing with slices of len==20 of real fasta files
First two cells, I read in files and then slice.
Second to last cell, I am trying to run it but get an error.

In [32]:
#!/usr/bin/env python3
#Biopython example for Blosum62
import sys
!{sys.executable} -m pip install 'biopython'
from Bio import pairwise2
from Bio import SeqIO
from Bio.Align import substitution_matrices
blosum62=substitution_matrices.load("BLOSUM62")
seq1 = SeqIO.read("example_fasta_files/homo_sapiens_lactate.fasta", "fasta")
seq2 = SeqIO.read("example_fasta_files/mus_musculus_lactate.fasta", "fasta")
# cast to string
a_str = str(seq1.seq)
b_str = str(seq1.seq)

In [33]:
print(f'{a_str[:20]}\n{b_str[20:40]}')
slice_a = a_str[:20]
slice_b = b_str[20:40]

MATLKDQLIYNLLKEEQTPQ
NKITVVGVGAVGMACAISIL


In [34]:
import numpy as np


Compare two input symbols

In [35]:
def compare(a, b, match, gap, mis_match):
    if a == b:
        return match
    elif " " == a or b == " ":
        return gap
    else:
        return mis_match

We build the matrics, the intake variable are two protein sequences, match score, mismatch score, and gap penalty. 
first the we need to initialize the matrix based on the length of two sequences. 
Then we calculate 

In [36]:
#currently the matrix is looking for minimum number, thus the scoreig
def build_matrics(seq1, seq2, match, mis_match, gap):
    a = len(seq1)
    b = len(seq2)
    
    #initialize the matrics
    S = np.zeros((a + 1, b + 1))
    
    #set 0 to the first row and the first column, or we can initialize the number from 1 to len(sequence)
    #as mentioned in https://en.wikipedia.org/wiki/Needleman%E2%80%93Wunsch_algorithm
    
    S[0, :] = np.fromfunction(lambda x, y: gap * (x + y), (1, b + 1), dtype=int)
    S[:, 0] = np.fromfunction(lambda x, y: gap * (x + y), (1,a + 1), dtype=int)
    
    #add empty space before the sequece.
    seq1 = " " + seq1[:]
    seq2 = " " + seq2[:]
    
    
    #longer length sequence set as row
    for i in range(1, b+1 if a>b else a+1):
        #iterate loop to find the minimum score.
        for j in range(i, b+1):
            last_score = [S[i - 1, j], S[i - 1, j - 1], S[i, j - 1]]
            change_score = [gap,compare(seq1[i],seq2[j], match, gap, mis_match),gap]

            new_score = np.add(last_score, change_score)

            S[i,j] = max(new_score)
            # print(S[i,c])
        for j in range(i+1, a+1):

            last_score = [S[j-1,i], S[j-1, i-1], S[j, i - 1]]
            change_score = [gap,compare(seq1[j],seq2[i], match, gap, mis_match),gap]
            new_score = np.add(last_score, change_score)
            S[j,i] = max(new_score)
    return S


In [37]:
#needs document(will do)
def trace_back(seq1, seq2, match, mis_match, gap, S, current_x, current_y, S1, S2, t1, t2):
    
    if current_x == 1 and current_y == 1:
        S1.append(seq1[1] + t1[:])
        S2.append(seq2[1] + t2[:])
        return
    if S[current_x,current_y]-S[current_x-1,current_y] == gap:
        trace_back(seq1, seq2, match, mis_match, gap, S, current_x-1, current_y, S1, S2, seq1[current_x]+t1[:], "-"+t2[:])
    if S[current_x,current_y]-S[current_x,current_y-1] == gap:
        trace_back(seq1, seq2, match, mis_match, gap, S, current_x, current_y-1, S1, S2, "-" + t1[:], seq2[current_y] + t2[:])
    if S[current_x,current_y]-S[current_x-1,current_y-1] == compare(seq1[current_x], seq2[current_y], match, gap, mis_match):
        trace_back(seq1, seq2, match, mis_match, gap, S, current_x-1, current_y-1, S1, S2, seq1[current_x] + t1[:], seq2[current_y] + t2[:])

In [38]:

F=build_matrics(slice_a, slice_b, 3,0,-1)
S1 = []
S2 = []
t1 = ""
t2 = ""
trace_back(" "+slice_a, " "+slice_b, 3, 0, -1, F, F.shape[0]-1, F.shape[1]-1, S1, S2, t1, t2)
print(S1)
print(S2)
print(F)

['MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ', 'MA-TLKDQLIYNLLKEEQTPQ', 'M-ATLKDQLIYNLLKEEQTPQ']
['NKITVVGVGAVGMACAISIL-', 'NKITVVGVGAVGMACAISIL-', 'NKITVVGVGAVGMACAISI-L', 'NKITVVGVGAVGMACAISI-L', 'NKITVVGVGAVGMACAIS-IL', 'NKITVVGVGAVGMACAIS-IL'

In [39]:
F = build_matrics("ATGGC","ACTG",3,-1,-1)
S1 = []
S2 = []
t1 = ""
t2 = ""
trace_back(" ATGGC", " ACTG", 3, -1, -1, F, F.shape[0]-1, F.shape[1]-1, S1, S2, t1, t2)
print(S1)
print(S2)
print(F)


['A-TGGC', 'A-TGGC']
['ACTG--', 'ACT-G-']
[[ 0. -1. -2. -3. -4.]
 [-1.  3.  2.  1.  0.]
 [-2.  2.  2.  5.  4.]
 [-3.  1.  1.  4.  8.]
 [-4.  0.  0.  3.  7.]
 [-5. -1.  3.  2.  6.]]
